# Simple RAG Training Pipeline

This notebook provides a streamlined 4-step process to generate synthetic QA data and launch a training job:

1. **Point to dataset** - Chunk and upload documents
2. **Create synthetic QA** - Generate question-answer pairs
3. **Create env** - Load search environment
4. **Run training job** - Launch the training


## Setup

Install dependencies and configure API credentials.


In [ ]:
# For Google Colab - clone repo and install
# !git clone https://github.com/cgftinc/synthetic-data-prep-lib.git
# %cd synthetic-data-prep-lib
# %pip install -e .[dev]

In [1]:
# Local development setup
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

repo_root = Path.cwd().parent
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [2]:
from synthetic_data_prep.utils import ensure_safe_python_version

ensure_safe_python_version()

In [5]:
# Configuration

# create an API key from the cgft console (https://app.cgft.io/account/api-keys).
API_KEY = "sk_LzDaFiZQhHnkWCDzAGFxaQoTEJOysfjwAbawWtKYAqUPzyfkdeHYJRbXiLsXhPkD"
BASE_URL = "https://app.cgft.io"

# Dataset configuration

# this should point to a local directory containing the documents you want to use for dataset generation.
# go to docs.cgft.io/rag/synthetic_datagen for sample documents you can use.
DOCS_PATH = "./samples/posthog/"
# this is the name of the corpus that will be created on the cgft platform to store your documents and generated dataset.
CORPUS_NAME = "posthog"

# QA generation configuration
# the following parameters are used to configure the synthetic question answer generation process. you can experiment with different values to see how they affect the generated dataset. the numbers below are just suggestions to get you started.
CORPUS_DESCRIPTION = "Posthog documentation including docs, company policy, etc."
EXAMPLE_QUERIES = [
    "how to feature flag",
    "setup reverse proxy to avoid ad blockers",
    "posthog install nextjs",
]
# Number of single-hop and multi-hop questions to generate. Single-hop questions can be answered with a single chunk, while multi-hop questions require reasoning over multiple chunks.
NUM_SINGLE_HOP = 40
NUM_MULTI_HOP = 5

# Optional: name for your training experiment
EXPERIMENT_NAME = "posthog search v1"

## Step 1: Point to Dataset

Chunk your documents and upload to corpus API in one line.


In [6]:
from synthetic_data_prep.corpus.corpora.source import CorporaChunkSource

source = CorporaChunkSource(api_key=API_KEY, corpus_name=CORPUS_NAME, base_url=BASE_URL)
source.populate_from_folder(DOCS_PATH)

Chunking documents from ./samples/posthog/...
ChunkCollection Summary
Total chunks: 3225
Total files: 1183

Chunk Size Statistics:
  Min: 29 chars (contribute/badge.md, chunk 0)
  Max: 2048 chars (data-warehouse/query.mdx, chunk 0)
  Mean: 1148 chars

File Structure:
├── data-warehouse/
│   ├── sql/
│   │   ├── index.mdx (8 chunks)
│   │   ├── useful-functions.mdx (7 chunks)
│   │   ├── variables.mdx (2 chunks)
│   │   └── data-access.mdx (3 chunks)
│   ├── views/
│   │   ├── index.mdx (2 chunks)
│   │   └── materialize.mdx (2 chunks)
│   ├── _snippets/
│   │   └── query-join-example.mdx (1 chunks)
│   ├── sources/
│   │   ├── stripe.mdx (1 chunks)
│   │   ├── tiktok-ads.mdx (1 chunks)
│   │   ├── azure-blob.mdx (1 chunks)
│   │   ├── attio.mdx (1 chunks)
│   │       ... 31 more files
│   ├── under-the-hood.md (1 chunks)
│   ├── query.mdx (4 chunks)
│   ├── tutorials.mdx (1 chunks)
│   ├── changelog.mdx (1 chunks)
│       ... 5 more files
├── how-posthog-works/
│   ├── clickhouse.md (3

Uploading chunks:   0%|          | 0/33 [00:00<?, ?batch/s]


Upload complete! Inserted: 3225


## Step 2: Create Synthetic QA

Generate synthetic question-answer pairs from your corpus. This will take a few minutes, so go get a coffee :)


In [7]:
from synthetic_data_prep.qa_generation.pipeline import generate_dataset

train_data, val_data = generate_dataset(
    source=source,
    api_key=API_KEY,
    corpus_description=CORPUS_DESCRIPTION,
    example_queries=EXAMPLE_QUERIES,
    num_single_hop=NUM_SINGLE_HOP,
    num_multi_hop=NUM_MULTI_HOP,
)

Generating corpus summary and example queries...

Generating 40 single-hop QA pairs...


Processing prompts:   0%|          | 0/40 [00:00<?, ?it/s]

Single-hop: 36 data points

Generating 5 multi-hop QA pairs...
Step 1: Generating related queries for 5 chunks...


Processing prompts:   0%|          | 0/5 [00:00<?, ?it/s]


Step 2: Validating 75 chunk pairs for multi-hop QA...


Processing prompts:   0%|          | 0/75 [00:00<?, ?it/s]

Multi-hop: 62 data points

Combined dataset: 98 total data points

Saved datasets:
  Train (68): outputs/train_dataset.yaml, outputs/train_dataset.jsonl
  Eval (30):  outputs/eval_dataset.yaml, outputs/eval_dataset.jsonl


## Step 3: Launch Training

Upload datasets and environment, then launch the training job.

In [ ]:
import synthetic_data_prep
from synthetic_data_prep.envs.cgft_search_env import CgftSearchEnv
from synthetic_data_prep.trainer.pipeline import train

experiment_id = train(
    env_class=CgftSearchEnv,
    env_args={
        "api_key": API_KEY,
        "corpus_id": source.corpus_id,
        "base_url": BASE_URL,
    },
    train_dataset=train_data,
    eval_dataset=val_data,
    prefix="cgft-search",
    api_key=API_KEY,
    base_url=BASE_URL,
    local_modules=[synthetic_data_prep],
    experiment_name=EXPERIMENT_NAME,
)

## Monitoring Training: What to Expect

### Early Training Noise

**Early training**: At the start of a training run, rewards will fluctuate and metrics may be noisy. This is completely normal - the model is still learning basic patterns and its outputs are unstable. Give it some time (usually a few dozen steps) and the signal will clean up and you'll start seeing clear trends.

**Exploration before exploitation**: Ideally, you want to see pass@k climbing first before average reward increases significantly. This means your model is exploring the solution space and learning to solve more prompts (breadth) before it optimizes for higher rewards on those prompts (depth). If average reward shoots up while pass@k stays low, you're likely exploiting a narrow set of easy prompts rather than building robust capabilities.

**Healthy training trajectory**: In a well-configured training run, expect pass@k to increase early as the model learns to solve more distinct prompts. Average reward should follow but lag behind. Eventually both should plateau as the model saturates your training distribution.

**Warning signs**:

- pass@k flat while average reward increases → model is overfitting to a narrow subset of prompts
- both metrics stagnate early → training distribution may be too hard, reward signal too sparse
